In [1]:
import csv


class DataTable:
    def __init__(self, csv_path: str,reference=None):
        """
        Load CSV data into memory.
        Convert all values to float.
        """
        self.data = [] 
        self.column_names = [] #the header values
                
        with open(csv_path, 'r') as file: #csv_path is a placeholder 
            reader = csv.reader(file) #reader that can go through the file line by line
            #it comes from the csv import thingy 
            self.column_names = next(reader) #skiping the header
            for row in reader:
                numbers = []
                for value in row:
                    if value == '?':
                        numbers.append(None)
                    else:
                        numbers.append(float(value))
                self.data.append(numbers)
         # Replace missing values (None) with a column mean
        for column_index in range(len(self.column_names)):
            if reference is None:
                # historical data: mean of this column's valid values
                valid = [row[column_index] for row in self.data if row[column_index] is not None]
                mean = sum(valid) / len(valid)
            else:
                # new cases: mean from the historical table
                mean = reference.get_column_average(self.column_names[column_index])

            for row in self.data:
                if row[column_index] is None:
                    row[column_index] = mean    


        

    def get(self, row_idx: int, col_name: str):
        """
        Return a single value from a row and column.
        """
        column_index = self.column_names.index(col_name) #finds the position index number of the column
        
        return self.data[row_idx][column_index] #the row_idex gets the whole row 
        #column_index gets the value of the index at that specific row
        

    def get_column(self, col_name: str) -> list:
        """
        Return all values in a column.
        """
        column_index = self.column_names.index(col_name) #find the index in the column like before 
        values = [] 
        for row in self.data: #looking through all the rows 
            values.append(row[column_index]) #grabbin the value from that column 

        return values

    def get_row(self, row_idx: int) -> dict:
        """
        Return a row as {column_name: value}
        """
       
        row = self.data[row_idx]
        result = {} #empty dictionary to fill up 

        for i in range(len(self.column_names)):
            col_name = self.column_names[i]
            result[col_name] = row[i] #adds a key-value pair to the dictionary
        return result
         
    

    def get_column_names(self) -> list:
        """
        Return list of column names.
        """
        return self.column_names # we are using the list from the innit functiion here 


    def get_column_average(self, col_name: str) -> float:
        """
        Return average of a column.
        """
        values = self.get_column(col_name)
        if values:
            return sum(values) / len(values)  
        else:
            return 0 #taking the average sum diviide by the length 

    def get_column_min(self, col_name: str) -> float:
        """
        Return minimum value of a column.
        """
        values = self.get_column(col_name)
        if values: 
            return min(values) 
        else:
            return 0

    def get_column_max(self, col_name: str) -> float:
        """
        Return maximum value of a column.
        """
        values = self.get_column(col_name)
        if values:
            return max(values)  
        else:
            return 0

In [2]:
from abc import ABC, abstractmethod
import math

from exceptions import OutOfSampleError


class Predictor(ABC):

    def __init__(self, historical_data):
        """
        Store historical DataTable.
        """
        self.historical_data = historical_data
        self.features = historical_data.get_column_names()[:-1]  # Feature1-Feature9

    def check_range(self, patient):
        """
        Raise OutOfSampleError if a feature is outside the historical min/max.
        """
        for f in self.features:
            low = self.historical_data.get_column_min(f)
            high = self.historical_data.get_column_max(f)
            if patient[f] < low or patient[f] > high:
                raise OutOfSampleError(
                    f"{f} with value {patient[f]} is outside historical range [{low}, {high}]"
                )

    @abstractmethod
    def predict(self, patient):
        """
        Make a prediction for one patient.
        """
        pass


class MajorityClassPredictor(Predictor):

    def __init__(self, historical_data):
        super().__init__(historical_data)
        self.majority_class = None
        labels = historical_data.get_column("TumorType")
        if labels.count(1.0) > labels.count(0.0):
            self.majority_class = 1.0
        else:
            self.majority_class = 0.0

    def predict(self, patient):
        """
        Always return the most common historical TumorType.
        """
        self.check_range(patient)
        return self.majority_class


class NearestNeighborPredictor(Predictor):

    def __init__(self, historical_data):
        super().__init__(historical_data)

    def distance(self, patient1, patient2):
        """
        Calculate Euclidean distance using Feature1-Feature9 only.
        """
        total = 0
        for f in self.features:
            total += (patient1[f] - patient2[f]) ** 2
        return math.sqrt(total)

    def predict(self, patient):
        """
        Find closest historical patient and return its TumorType.
        """
        self.check_range(patient)
        best_distance = None
        best_label = None
        for i in range(len(self.historical_data.data)):
            other = self.historical_data.get_row(i)
            d = self.distance(patient, other)
            if best_distance is None or d < best_distance:
                best_distance = d
                best_label = other["TumorType"]
        return best_label

In [3]:
class OutOfSampleError(Exception):

    # OutOfSampleError --> Name of custom exception 
    # Exception is the built in exception class that my custom class inherits from 


    
    """
    Raised when a new input contains feature values
    outside the historical min/max range.
    """
    pass 


# Feature4 with value 1.0 is outside historical range [0.1, 0.9]


In [4]:

from data_representation import DataTable
from predictors import MajorityClassPredictor, NearestNeighborPredictor
from exceptions import OutOfSampleError


def calculate_accuracy(predictions, actual):
    """Calculate accuracy using only successfully predicted cases."""
    if len(predictions) != len(actual):
        raise ValueError(
            "Predictions and actual lists must have equal lengths."
        )

    matched_predictions = 0
    valid_predictions = 0

    for prediction, actual_label in zip(predictions, actual):
        # None means the predictor could not predict this case.
        if prediction is None:
            continue

        valid_predictions += 1

        if prediction == actual_label:
            matched_predictions += 1

    if valid_predictions == 0:
        raise ValueError(
            "Cannot calculate accuracy without valid predictions."
        )

    return matched_predictions / valid_predictions


def main():
    # Load datasets.
    historical = DataTable("data/cancer_historical1.csv")

    new_cases = DataTable(
        "data/cancer_new_cases1.csv",
        reference=historical
    )

    labels = DataTable("data/cancer_new_cases_labels1.csv")

    # Get the true labels.
    actual_labels = labels.get_column("TumorType")

    # Ensure that each new case has a corresponding true label.
    if len(new_cases.data) != len(actual_labels):
        raise ValueError(
            "The number of new cases must match the number of labels."
        )

    # Create predictors.
    mode_predictor = MajorityClassPredictor(historical)
    nn_predictor = NearestNeighborPredictor(historical)

    mode_predictions = []
    nn_predictions = []

    # Predict every new case, preserving row alignment.
    for i in range(len(new_cases.data)):
        patient = new_cases.get_row(i)

        # Test the majority-class predictor independently.
        try:
            mode_prediction = mode_predictor.predict(patient)
        except OutOfSampleError as error:
            print(f"Mode, case {i}: {error}")
            mode_prediction = None

        # Test the nearest-neighbor predictor independently.
        try:
            nn_prediction = nn_predictor.predict(patient)
        except OutOfSampleError as error:
            print(f"Nearest Neighbor, case {i}: {error}")
            nn_prediction = None

        # Append one result per case to both lists.
        mode_predictions.append(mode_prediction)
        nn_predictions.append(nn_prediction)

    # Report accuracy and prediction coverage.
    for name, predictions in [
        ("Mode", mode_predictions),
        ("Nearest Neighbor", nn_predictions),
    ]:
        valid_count = sum(
            prediction is not None for prediction in predictions
        )

        print(
            f"{name} valid predictions: "
            f"{valid_count}/{len(actual_labels)}"
        )

        if valid_count == 0:
            print(f"{name} accuracy: unavailable")
        else:
            accuracy = calculate_accuracy(
                predictions, actual_labels
            )
            print(f"{name} accuracy: {accuracy:.4f}")


if __name__ == "__main__":
    main()


Mode valid predictions: 100/100
Mode accuracy: 0.7900
Nearest Neighbor valid predictions: 100/100
Nearest Neighbor accuracy: 0.9800


In [5]:


import csv
import os
import tempfile

from data_representation import DataTable
from predictors import (
    MajorityClassPredictor,
    NearestNeighborPredictor
)
from exceptions import OutOfSampleError


def write_test_csv(path, headers, rows):
    """Write a small CSV file for testing."""
    with open(path, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(headers)
        writer.writerows(rows)


def run_test(name, test_function):
    """Run a test and report its result."""
    try:
        test_function()
        print(f"PASS: {name}")
    except Exception as error:
        print(
            f"FAIL: {name} — "
            f"{type(error).__name__}: {error}"
        )

In [6]:
test_folder = tempfile.TemporaryDirectory()

historical_path = os.path.join(
    test_folder.name, "historical.csv"
)

write_test_csv(
    historical_path,
    ["Feature1", "Feature2", "TumorType"],
    [
        [0, 0, 0],
        [10, 10, 1],
        [2, 2, 0],
    ]
)

historical = DataTable(historical_path)

print("Historical test data loaded successfully.")
print("Column names:", historical.get_column_names())
print("Number of rows:", len(historical.data))

Historical test data loaded successfully.
Column names: ['Feature1', 'Feature2', 'TumorType']
Number of rows: 3


Normal Case

In [7]:
# Normal Case

# Test 1: Retrieve one value.
def test_get():
    assert historical.get(1, "Feature1") == 10.0


# Test 2: Retrieve a complete column.
def test_get_column():
    assert historical.get_column("Feature1") == [
        0.0, 10.0, 2.0
    ]


# Test 3: Retrieve a complete row as a dictionary.
def test_get_row():
    expected = {
        "Feature1": 10.0,
        "Feature2": 10.0,
        "TumorType": 1.0
    }

    assert historical.get_row(1) == expected


# Test 4: Retrieve column names.
def test_get_column_names():
    assert historical.get_column_names() == [
        "Feature1", "Feature2", "TumorType"
    ]


# Test 5: Calculate a column average.
def test_column_average():
    assert historical.get_column_average("Feature1") == 4.0


# Test 6: Calculate the column minimum.
def test_column_min():
    assert historical.get_column_min("Feature1") == 0.0


# Test 7: Calculate the column maximum.
def test_column_max():
    assert historical.get_column_max("Feature1") == 10.0


run_test("DataTable.get()", test_get)
run_test("DataTable.get_column()", test_get_column)
run_test("DataTable.get_row()", test_get_row)
run_test("DataTable.get_column_names()", test_get_column_names)
run_test("DataTable.get_column_average()", test_column_average)
run_test("DataTable.get_column_min()", test_column_min)
run_test("DataTable.get_column_max()", test_column_max)

PASS: DataTable.get()
PASS: DataTable.get_column()
PASS: DataTable.get_row()
PASS: DataTable.get_column_names()
PASS: DataTable.get_column_average()
PASS: DataTable.get_column_min()
PASS: DataTable.get_column_max()


Special Case

In [8]:
# Create a historical dataset containing a missing value.
missing_historical_path = os.path.join(
    test_folder.name, "historical_missing.csv"
)

write_test_csv(
    missing_historical_path,
    ["Feature1", "Feature2", "TumorType"],
    [
        [2, 10, 0],
        [4, "?", 1],
        [6, 30, 0],
    ]
)

missing_historical = DataTable(missing_historical_path)


# Test 8: Historical missing value is replaced by its column mean.
def test_historical_imputation():
    # Feature2 mean = (10 + 30) / 2 = 20.
    assert missing_historical.get(1, "Feature2") == 20.0


# Create new cases with a missing Feature1.
missing_new_path = os.path.join(
    test_folder.name, "new_cases_missing.csv"
)

write_test_csv(
    missing_new_path,
    ["Feature1", "Feature2"],
    [
        ["?", 15],
    ]
)

new_cases_test = DataTable(
    missing_new_path,
    reference=missing_historical
)


# Test 9: New cases use the historical Feature1 average.
def test_new_case_imputation():
    # Historical Feature1 mean = (2 + 4 + 6) / 3 = 4.
    assert new_cases_test.get(0, "Feature1") == 4.0


# Test 10: Feature values exactly at their historical
# minimum and maximum should be accepted.
def test_boundary_values():
    predictor = NearestNeighborPredictor(historical)

    minimum_patient = {
        "Feature1": 0.0,
        "Feature2": 0.0
    }

    maximum_patient = {
        "Feature1": 10.0,
        "Feature2": 10.0
    }

    predictor.check_range(minimum_patient)
    predictor.check_range(maximum_patient)


run_test(
    "Historical missing-value imputation",
    test_historical_imputation
)

run_test(
    "New cases use historical averages",
    test_new_case_imputation
)

run_test(
    "Historical minimum and maximum boundaries",
    test_boundary_values
)

PASS: Historical missing-value imputation
PASS: New cases use historical averages
PASS: Historical minimum and maximum boundaries


Error Case

In [9]:
predictor = NearestNeighborPredictor(historical)


# Test 11: A feature below its historical minimum.
def test_below_minimum():
    patient = {
        "Feature1": -1.0,
        "Feature2": 5.0
    }

    try:
        predictor.predict(patient)
    except OutOfSampleError as error:
        assert "Feature1" in str(error)
        print("Actual error:", error)
    else:
        raise AssertionError(
            "Expected OutOfSampleError below the minimum."
        )


# Test 12: A feature above its historical maximum.
def test_above_maximum():
    patient = {
        "Feature1": 5.0,
        "Feature2": 11.0
    }

    try:
        predictor.predict(patient)
    except OutOfSampleError as error:
        assert "Feature2" in str(error)
        print("Actual error:", error)
    else:
        raise AssertionError(
            "Expected OutOfSampleError above the maximum."
        )


run_test("Out-of-sample value below minimum", test_below_minimum)
run_test("Out-of-sample value above maximum", test_above_maximum)

Actual error: Feature1 with value -1.0 is outside historical range [0.0, 10.0]
PASS: Out-of-sample value below minimum
Actual error: Feature2 with value 11.0 is outside historical range [0.0, 10.0]
PASS: Out-of-sample value above maximum


Normal Cases: Test Both Predictors

In [10]:
# Test 13: Majority-class predictor.
def test_majority_predictor():
    predictor = MajorityClassPredictor(historical)

    patient = {
        "Feature1": 5.0,
        "Feature2": 5.0
    }

    # Historical labels are [0, 1, 0].
    # The majority class is 0.
    assert predictor.predict(patient) == 0.0


# Test 14: Nearest-neighbor predictor.
def test_nearest_neighbor_predictor():
    predictor = NearestNeighborPredictor(historical)

    patient = {
        "Feature1": 9.0,
        "Feature2": 9.0
    }

    # The nearest historical patient has features (10, 10)
    # and TumorType = 1.
    assert predictor.predict(patient) == 1.0


run_test("Majority-class prediction", test_majority_predictor)
run_test("Nearest-neighbor prediction", test_nearest_neighbor_predictor)

PASS: Majority-class prediction
PASS: Nearest-neighbor prediction


Error Cases: Testing Accuracy Function

In [11]:
from main import calculate_accuracy


# Test 15: Normal accuracy calculation.
def test_accuracy_normal():
    predictions = [0.0, 1.0, 1.0]
    actual = [0.0, 0.0, 1.0]

    # Two of three predictions are correct.
    assert abs(
        calculate_accuracy(predictions, actual) - (2 / 3)
    ) < 1e-10


# Test 16: Accuracy with an unpredicted case.
def test_accuracy_with_none():
    predictions = [0.0, None, 1.0]
    actual = [0.0, 1.0, 1.0]

    # Two valid predictions, both correct.
    expected_accuracy = 1.0

    actual_accuracy = calculate_accuracy(
        predictions, actual
    )

    assert actual_accuracy == expected_accuracy


# Test 17: Unequal list lengths.
def test_accuracy_unequal_lengths():
    try:
        calculate_accuracy([0.0], [0.0, 1.0])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError for unequal list lengths."
        )


# Test 18: No predictions.
def test_accuracy_empty():
    try:
        calculate_accuracy([], [])
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected ValueError when no predictions are available."
        )


run_test("Accuracy with ordinary predictions", test_accuracy_normal)
run_test("Accuracy with a missing prediction", test_accuracy_with_none)
run_test("Accuracy with unequal list lengths", test_accuracy_unequal_lengths)
run_test("Accuracy with empty lists", test_accuracy_empty)

PASS: Accuracy with ordinary predictions
PASS: Accuracy with a missing prediction
PASS: Accuracy with unequal list lengths
PASS: Accuracy with empty lists


Run Complete Program

In [12]:
import main
main.main()



Mode valid predictions: 100/100
Mode accuracy: 0.7900
Nearest Neighbor valid predictions: 100/100
Nearest Neighbor accuracy: 0.9800
